# Outcome-only RLVR — seed 0 (dense warm start)

STEP-1 §6's optional hybrid row: outcome-only GRPO applied to the completed dense seed-0
checkpoint. Attach that dense output as a Kaggle input first. Any capability here is partly the
dense teacher's; it does not substitute for the from-initialization arm.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, urllib.request
REPO_URL='https://github.com/escher-bach/actuallybuildingstuff.git'
GIT_COMMIT='__FINAL_COMMIT_SHA__'
CONFIG_REL='step1/configs/kaggle/t4x2_rlvr_warmstart_seed0.toml'
WORKING=Path('/kaggle/working'); SOURCE=WORKING/'actuallybuildingstuff'; PROJECT=SOURCE/'baby-llm-foundations'; OUTPUT=WORKING/'rlvr-warmstart-seed0'
assert len(GIT_COMMIT)==40 and all(c in '0123456789abcdef' for c in GIT_COMMIT)
assert not SOURCE.exists(), f'fresh batch session required: {SOURCE}'

In [ ]:
env=os.environ.copy(); env.update({'GIT_TERMINAL_PROMPT':'0','PYTHONUNBUFFERED':'1','PIP_DISABLE_PIP_VERSION_CHECK':'1','WANDB_MODE':'disabled','TOKENIZERS_PARALLELISM':'false'})
subprocess.run(['git','clone',REPO_URL,str(SOURCE)],check=True,env=env); subprocess.run(['git','-C',str(SOURCE),'checkout','--detach',GIT_COMMIT],check=True,env=env)
assert subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True,env=env).strip()==GIT_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-r',str(PROJECT/'requirements-kaggle.txt')],check=True,env=env)
if not shutil.which('cargo'):
    rustup=WORKING/'rustup-init'; urllib.request.urlretrieve('https://static.rust-lang.org/rustup/dist/x86_64-unknown-linux-gnu/rustup-init',rustup); rustup.chmod(0o755)
    subprocess.run([str(rustup),'-y','--profile','minimal','--default-toolchain','1.85.0'],check=True,env=env); env['PATH']=str(Path.home()/'.cargo/bin')+os.pathsep+env['PATH']
subprocess.run([sys.executable,'-m','maturin','build','--release','--manifest-path',str(PROJECT/'step1/crates/world-py/Cargo.toml')],cwd=str(PROJECT/'step1'),check=True,env=env)
wheel=sorted((PROJECT/'step1/target/wheels').glob('world_py-*.whl'))[-1]; subprocess.run([sys.executable,'-m','pip','install','--force-reinstall',str(wheel)],check=True,env=env)

In [ ]:
sys.path.insert(0,str(PROJECT/'step1/python'))
from step1_experiments.rlvr import load_rlvr_config
from step1_experiments.transfer import locate_dense_source
config=load_rlvr_config(PROJECT/CONFIG_REL); DENSE_SOURCE=locate_dense_source([Path('/kaggle/input')],config['source']); print('Using exact dense source:',DENSE_SOURCE)
cmd=['torchrun','--standalone','--nproc_per_node=2','-m','step1_experiments.rlvr','--config',str(PROJECT/CONFIG_REL),'--output-dir',str(OUTPUT)]
cmd+=['--source-run',str(DENSE_SOURCE)]
subprocess.run(cmd,cwd=str(PROJECT/'step1/python'),env=env,check=True)

In [ ]:
import json, math
report=json.loads((OUTPUT/'rlvr_report.json').read_text())
assert report['contract']=='step1_rlvr_grpo_v1'
assert report['algorithm']['trainer']=='trl.GRPOTrainer' and report['algorithm']['signal']=='outcome_only_privileged_verifier'
assert report['algorithm']['privileged_intermediate_labels'] is False and report['algorithm']['reward_weights'][0]==1.0
budget=report['budget_accounting']
assert budget['optimizer_updates']==191 and budget['rollout_episodes']==12224
assert [m['budget_updates'] for m in report['milestones']]==[48, 95, 191]
assert all(m['serialization']['exact'] for m in report['milestones'])
assert all(v is None or math.isfinite(v) for m in report['milestones'] for s in m['evaluation'].values() for v in s['metrics'].values())
assert report['initialization']['policy']=='dense_A_checkpoint'
assert report['initialization']['source_git_sha']==config['source']['git_sha']
signal=report['training_signal']
print('groups with reward variance:',signal['updates_with_any_reward_variance'],'of',signal['logged_updates'],
      '| mean frac_reward_zero_std:',signal['frac_reward_zero_std_mean'])
print(json.dumps({'budget':budget,'training_signal':signal,
  'milestones':[{'budget_updates':m['budget_updates'],'evaluation':{k:v['metrics'] for k,v in m['evaluation'].items()}} for m in report['milestones']]},indent=2))